# MCP (Model Context Protocol)

## Why this matters

You're already a user of MCP without necessarily having the internals named -- the Notion, Gmail, Google Calendar, and Supabase connectors available in this environment are MCP servers, and the earlier notebook that mentioned "MCP primitives studied (tools vs. resources vs. prompts)" from the Databricks/platform learning was the surface-level version of this topic. This notebook goes deeper: what MCP actually standardizes, why it exists as a protocol rather than everyone building bespoke integrations, and where it sits relative to the tool-use mechanism already covered in the structured-output notebook -- because MCP and tool-use are related but answer different questions, and conflating them is a common confusion.

## Level 1: What it is

**MCP** is an open protocol (originated by Anthropic, since adopted more broadly) that standardizes how an LLM application connects to external data sources and tools. Before MCP, every application wanting to connect an LLM to, say, Notion, Gmail, and a Postgres database would write three separate bespoke integrations, each with its own auth flow, its own way of describing available actions, its own error handling conventions. MCP defines a single client-server protocol so that:

- An **MCP server** exposes a data source or tool (Notion, Gmail, a database, a filesystem) through a standard interface
- An **MCP client** (the LLM application -- Claude Desktop, Claude Code, this very chat interface) can connect to *any* MCP server without needing custom integration code per server
- The same MCP server built once can be used by any MCP-compatible client, not just the one it was originally built for

MCP defines three primitive types a server can expose: **tools** (actions the model can invoke, like `notion-create-pages`), **resources** (data the client can read, like file contents or a document), and **prompts** (reusable prompt templates the server provides).

## Level 2: How it actually works internally

### MCP vs. tool-use: two different layers, easy to conflate

This is the distinction worth being precise about. **Tool-use** (covered in the structured-output notebook) is the mechanism by which *a model call* can request a function be executed and get a structured result back -- it's about the shape of a single request/response cycle with the model. **MCP** is a layer *above* that: it's how an *application* discovers what tools/resources are available from a *server* in the first place, and how it manages the connection, authentication, and session with that server. A tool-use call still happens underneath, once MCP has told the application what tools exist and how to describe them to the model -- MCP doesn't replace tool-use, it standardizes *where the tool definitions come from* and *how the actual execution gets routed* to the right external system.

Concretely, in this environment: when a Notion MCP server is connected, its available actions (`notion-search`, `notion-create-pages`, etc.) get surfaced as tool definitions the model can call via ordinary tool-use, exactly like the `ISSUE_TOOL` schema in `base_agent.py`. What MCP adds on top is the discovery mechanism (the client didn't need custom code to know Notion's tools exist and what their schemas are) and the routing (when the tool is called, MCP handles getting that call to the actual Notion API and the result back), not the tool-use call format itself.

### The client-server architecture

1. **MCP server**: a process that exposes a defined set of tools/resources/prompts over the protocol (typically JSON-RPC 2.0 messages over a transport like stdio, for local servers, or HTTP/SSE for remote ones). The server owns the actual connection to the underlying system (Notion's API, a database, a filesystem) and translates between MCP's standard format and that system's specifics.
2. **MCP client**: embedded in the host application (Claude Desktop, an IDE plugin, this chat interface). It connects to one or more MCP servers, queries them for their available tools/resources, and surfaces those to the model as part of the conversation -- functionally becoming additional tool-use options alongside anything else the application already supports.
3. **Discovery and capability negotiation**: on connecting, the client asks the server what it supports (which tools, which resources, which prompts), rather than that being hardcoded ahead of time -- this is what lets the *same* client work with servers it's never seen before, as long as they speak MCP.

### Why a protocol, not just "more tools"

The alternative to a standard protocol is N applications each writing M bespoke integrations -- N×M pieces of integration code, each maintained separately, each potentially handling auth/errors/pagination differently. MCP turns that into N clients + M servers, where any client can use any server: write a Notion MCP server once, and every MCP-compatible application (not just the one that originally built it) can use it. This is the same shape of problem USB solved for hardware peripherals, or ODBC solved for database drivers -- a standard interface so N×M bespoke integrations collapse to N+M standard ones.

## Level 3: Where it breaks, and what it doesn't solve

1. **MCP doesn't make an external system's own API any more reliable.** If the underlying service (Notion, Gmail) has rate limits, auth expiry, or flaky uptime, MCP faithfully surfaces those problems -- it standardizes the interface, not the reliability of what's behind it. A failed MCP tool call because of an underlying auth/credential error is still a real failure to handle, not something the protocol papers over (this is exactly why this environment's tooling explicitly calls out re-authentication flows for failed MCP tool calls).

2. **Trust boundary: an MCP server is a third party with real capability, not just a passive data source.** Connecting an MCP server that can, say, send emails or modify documents means giving a piece of external code the ability to take real actions triggered by model output. This is precisely why environments built on MCP need explicit rules about which actions require user confirmation before executing (send, delete, publish, modify) versus which are safe to run without asking (read-only lookups) -- MCP as a protocol doesn't itself enforce this distinction; it has to be built into how the client uses it.

3. **Prompt injection through MCP server content is a real, documented risk, not a hypothetical.** If an MCP server surfaces content from an untrusted source (a document someone else wrote, a webpage), and that content contains text designed to look like instructions to the model, the model can be misled into treating retrieved content as commands rather than data. This is the same class of problem as the "prompt injection via code comments" limitation flagged in the X++ project's own README -- MCP doesn't introduce this risk, but it does mean more channels through which untrusted content can reach the model, so the discipline of treating tool/resource output as data-not-instructions matters more, not less, as more MCP servers get connected.

4. **Discovery doesn't mean the model will use tools well.** MCP solves "the model knows this tool exists and what its schema is" -- it doesn't solve "the model reliably picks the right tool, with the right arguments, at the right moment." That's still a prompting/judgment problem, same category as everything discussed about LLM-judgment variance in the X++ agents -- MCP expands the space of things a model *could* do, it doesn't make the model's choices among them more consistent.

5. **Not every MCP server implementation is equally well-built.** The protocol standardizes the interface shape, but a poorly-documented tool description (vague `description` field, unclear parameter semantics) still produces poor model behavior even over a technically-correct MCP connection -- same principle as a badly-written docstring not becoming useful just because the function signature is syntactically valid.

In [ ]:
# Conceptual sketch of what an MCP server's tool definition looks like
# on the wire -- this is illustrative of the shape, not a runnable
# client/server (a real MCP server needs the actual protocol library
# and a transport; this cell exists to make the "what gets exposed"
# concept concrete, matching the tool-use schema shape from the
# structured-output notebook).

# What an MCP server advertises when a client asks "what tools do you have":
mcp_server_tool_definition = {
    "name": "notion-create-pages",
    "description": "Creates one or more Notion pages with specified properties and content.",
    "inputSchema": {
        "type": "object",
        "properties": {
            "pages": {"type": "array", "items": {"type": "object"}},
        },
        "required": ["pages"],
    },
}

# This is structurally almost identical to the ISSUE_TOOL schema from
# base_agent.py -- same idea (name, description, JSON-schema-typed
# input), because MCP tool definitions ARE tool-use tool definitions,
# just sourced from a server via the MCP discovery handshake instead
# of being hardcoded in your own Python file like ISSUE_TOOL is.
print("MCP tool definitions and hand-written tool-use schemas share the "
      "exact same underlying shape -- MCP just changes WHERE the "
      "definition comes from (a server, discovered at connect time) "
      "rather than HOW the model uses it once it has one.")

## Interview-ready summary

**Q: "What is MCP and how does it relate to function calling / tool use?"**

Weak answer: "MCP is how Claude connects to external tools like Notion or Gmail."

Strong answer: "Tool-use is the mechanism for a single model call to request a function execution and get a structured result back -- that's the request/response shape, and it's provider-level (Anthropic's tool-use API, OpenAI's function calling). MCP is a layer above that: an open, standardized client-server protocol for how an *application* discovers what tools and resources are available from a *server*, and manages that connection -- authentication, discovery, routing execution to the right backend. Without MCP, every application wanting to connect an LLM to Notion, Gmail, and a database would write bespoke integration code for each; MCP collapses that N-times-M problem into servers built once and usable by any MCP-compatible client. It doesn't replace tool-use -- the tool call underneath still happens the same way -- it standardizes where the tool definitions come from and how execution gets routed, and it introduces its own considerations around trust boundaries and prompt-injection risk that have to be handled at the application layer, not by the protocol itself."